## Qutip Tutorial

#### Imports

In [2]:
from __future__ import annotations
import qutip
from qutip import operators
from typing import List,Tuple,Optional,Type,Dict
import numpy as np

#### First: Define the transverse ising model 

In qutip the spin basis is simply a Fock representation with 2 D.O.F (Hardcore bosons). If we want to write the hamiltonian we should consider the tensor product operator for each term (it is not like quspin).

#### Define the general operator method

We implement a function in order to compute the general many-body operator

$O^{a_1,a_2,...,a_n}_{i_1, i_2, ..., i_n}=G^{a_1, a_2, .. a_n}_{i_1,i_2,...,i_n} s^{a_1}_{i_1} \otimes s^{a_2}_{i_2} \otimes ... \otimes s^{a_n}_{i_n}$

We study the operator $O_{i ,i+1}=\sum^{N-1}_i x_i \otimes x_{i+1}$

#### Implement a SteadyState class

In [123]:

class SteadyStateClass():
    
    def __init__(self,size:int,unitary_list:List[List],dissipative_list:List[List]) -> None:
        
        #parameters
        self.unitary_list:List[List]=unitary_list
        self.dissipative_list:List[List]=dissipative_list
        self.size=size
        
        #attributes
        self.steady_state:qutip.Qobj=None
        self.limbladian:qutip.Qobj=None
    
    
    #operation that convert the abstract string to the qutip.Qobj    
    def _manybodyoperator(self,directions:List[List],size:int)->qutip.Qobj:
        #pauli dictionary
        pauli={'idx':qutip.identity(2),'x':qutip.sigmax(),'y':qutip.sigmay(),'z':qutip.sigmaz(),'+':qutip.sigmap,'-':qutip.sigmam}

        # for each coupling term in the list direction
        for r,direction in enumerate(directions):
            coupling=direction[0] #coupling term in the direction list
            # starting point -> identity operator
            idx_mb:List[str]=['idx' for i in range(size)]
            
            # create the op representation
            for dir,i in direction[1:]:
                #print('dir=',dir,'i=',i)
                idx_mb[i]=dir
            # convert into qutip.Qobj
            for i in range(size):
                if i==0:
                    op=pauli[idx_mb[i]]        
                else:
                    op=qutip.tensor(op,pauli[idx_mb[i]])
            #sum each direction
            if r==0:
                manybodyop=op*coupling
            else:
                manybodyop=manybodyop+op*coupling
                    
        return manybodyop
    
    def _get_the_limbladian(self)->None:
        #define the hamiltonian
        for i,u in enumerate(self.unitary_list):
            if i==0:
                hamiltonian=self._manybodyoperator(directions=u,size=self.size)
            else:
                hamiltonian=hamiltonian+self._manybodyoperator(directions=u,size=self.size)
        dissipative=[]
        for d in self.dissipative_list:
            dissipative.append(self._manybodyoperator(directions=d,size=self.size))
        self.limbladian=qutip.liouvillian(H=hamiltonian,c_ops=dissipative)
        
    def get_steady_state(self)->qutip.Qobj:
        self._get_the_limbladian()
        self.steady_state=qutip.steadystate(qutip.to_super(self.limbladian))

    
    def print_liouvillian(self)->None:
        print('Unitary part=\n',self.unitary_list,'\n')
        print('Dissipative part=\n',self.dissipative_list,'\n')
        
    def steady_state_expect(self,directions:List[List])->float:
        # define the operator in 
        op=self._manybodyoperator(directions=directions,size=self.size)
        return qutip.expect(op,self.steady_state)        
    
    

#### Check if the SteadyState Class works

In [124]:
size:int=3
j:float=1
h:float=1
g:float=0.1
xx:List[List]=[[j,('x',i),('x',i+1)] for i in range(size-1)]
z:List[List]=[[h,('z',i)] for i in range(size)]
x:List[List]=[[g,('x',i)] for i in range(size)]

unitary=[xx,z]
dissipative=[]

std=SteadyStateClass(size=size,unitary_list=unitary,dissipative_list=dissipative)

std.print_liouvillian()

std.get_steady_state()

print(std.steady_state)

print(std.steady_state_expect(xx+z))





Unitary part=
 [[[1, ('x', 0), ('x', 1)], [1, ('x', 1), ('x', 2)]], [[1, ('z', 0)], [1, ('z', 1)], [1, ('z', 2)]]] 

Dissipative part=
 [] 

Quantum object: dims = [[2, 2, 2], [2, 2, 2]], shape = (8, 8), type = oper, isherm = True
Qobj data =
[[ 9.79125837e-07+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j -9.09081556e-02+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j -4.54542537e-02-1.35525272e-20j
  -9.09081556e-02+0.00000000e+00j  0.00000000e+00+0.00000000e+00j]
 [ 0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j]
 [ 0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e

#### Another method for the directions

In [42]:
op={}
index=(0,1,2)
coupling=1
direction=['x','x','x']
op[index]={"coupling":coupling, "direction":direction}

class AbstractOperator():
    def __init__(self,index:List[Tuple],direction:List[List],coupling:List,size:int) -> None:
        
        self.index=index
        self.direction=direction
        self.coupling=coupling
        self.size=size
        self.op:dict={}
            
        # if indices> size the operator is ill defined
        assert max([max(idx) for idx in self.index])<=(self.size-1),f"operator defined in a larger size system: idx > l={self.size}" 
        self.__get_operator()

    def __get_operator(self):
        for i,idx in enumerate(self.index):
            self.op[idx]={"coupling":self.coupling[i],"direction":self.direction[i]}
            
    def append(self,ao:AbstractOperator)->Optional[AbstractOperator]:
        assert self.size==ao.size,"size!=operator size"
        self.direction+=ao.direction
        self.index+=ao.index
        self.coupling+=ao.coupling
        self.__get_operator()
        
        return self
        
    def printout(self):
        print(self.op)
        
    def abstract2qutip(self)->Tuple[qutip.Qobj,List[qutip.Qobj]]:
        #operation that convert the abstract string to the qutip.Qobj    
        #pauli dictionary
        pauli={'idx':qutip.identity(2),'x':qutip.sigmax(),'y':qutip.sigmay(),'z':qutip.sigmaz(),'+':qutip.sigmap(),'-':qutip.sigmam()}

        qutip_op:qutip.Qobj=0
        qutip_op_density:Dict[qutip.Qobj]={}
        # create the op representation
        for index in self.op.keys():
            # starting point -> identity operator
            idx_mb:List[str]=['idx' for i in range(self.size)]
            for j,idx in enumerate(index):
                idx_mb[idx]=self.op[index]['direction'][j]
        # convert into qutip.Qobj
            for i in range(self.size):
                if i==0:
                    op=pauli[idx_mb[i]]        
                else:
                    op=qutip.tensor(op,pauli[idx_mb[i]])
            #sum each direction
            qutip_op=qutip_op+op*self.op[index]['coupling']
            qutip_op_density[index]=op*self.op[index]['coupling']
        return qutip_op,qutip_op_density
    
    def exp_value(self,psi:qutip.Qobj)->float:
        qutip_op,_=self.abstract2qutip()
        return qutip.expect(qutip_op,psi)
    
    def exp_value_density(self,psi:qutip.Qobj)->Dict[qutip.QObj]:
        _,qutip_op_density=self.abstract2qutip()
        values:dict={}
        for index in qutip_op_density.keys():
            values[index]=qutip.expect(qutip_op_density[index],psi)
        return values
    
    
# we still can implement new attributes
# such as eigsh and gs_state
class IsingHamiltonian(AbstractOperator):
    def __init__(self,j_coupling:Dict,direction_coupling:Tuple[str],ext_field:Dict,field_direction:str) -> None:
        
        self.len_couplings=len(list(j_coupling.keys()))
        sum_coupling=j_coupling | ext_field
        index=list(sum_coupling.keys())
        directions=[[direction_coupling[0],direction_coupling[1]] for k in j_coupling.keys()]+[[field_direction] for k in ext_field.keys()]
        size=len(ext_field)
        interaction_values=list(sum_coupling.values())
        
        super().__init__(index,directions,interaction_values,len(ext_field))
    
    def printout(self):
        
        print("Coupling Term: \n")
        print(list(self.op.items())[:self.len_couplings],'\n')
        print("External field: \n")
        print(list(self.op.items())[self.len_couplings:],'\n')

        
        

        
class SteadyStateSolver():
    
    def __init__(self,hamiltonian:AbstractOperator,dissipative_ops:List[AbstractOperator]) -> None:
        
        #parameters
        self.hamiltonian=hamiltonian
        self.dissipative_ops=dissipative_ops
        self.size=hamiltonian.size
        
        #attributes
        self.steady_state:qutip.Qobj=None
        self.limbladian:qutip.Qobj=None
    
    def __get_the_limbladian(self)->None:
        #define the hamiltonian
        hamiltonian_qutip=self.hamiltonian.abstract2qutip()[0]
        dissipative_qutip=[d.abstract2qutip()[0] for d in self.dissipative_ops]
        self.limbladian=qutip.liouvillian(H=hamiltonian_qutip,c_ops=dissipative_qutip)
        
    def get_steady_state(self)->qutip.Qobj:
        self.__get_the_limbladian()
        self.steady_state=qutip.steadystate(qutip.to_super(self.limbladian))

    
    def print_liouvillian(self)->None:
        print('Unitary part=\n')
        self.hamiltonian.printout()
        print('\n')
        print('Dissipative part=\n')
        for d in self.dissipative_ops:
            d.printout()
        print('\n')
        
    def steady_state_expect(self,op:AbstractOperator)->float:
        # define the operator in 
        return op.exp_value(self.steady_state)
    
    def steady_state_expect_density(self,op:AbstractOperator)->float:
        return op.exp_value_density()        
    

#### Example of AbstractOperator

In [43]:
l=2
index_xx=[(0,1)]
direction=[['x','x']]
coupling=[0.5]

xx=AbstractOperator(index=index_xx,direction=direction,coupling=coupling,size=l)

xx.printout()

xx_qutip=xx.abstract2qutip()[0]

print(xx_qutip)


{(0, 1): {'coupling': 0.5, 'direction': ['x', 'x']}}
Quantum object: dims = [[2, 2], [2, 2]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[0.  0.  0.  0.5]
 [0.  0.  0.5 0. ]
 [0.  0.5 0.  0. ]
 [0.5 0.  0.  0. ]]


#### Create the Spin Hamiltonian for Abstract operators

In [45]:
l=3
ad_j={}
h={}
for i in range(l):
    ad_j[(i%l,(i+1)%l)]=-1.        
    h[(i,)]=1.

#create the Spin Hamiltonian
hamiltonian=IsingHamiltonian(j_coupling=ad_j,direction_coupling=['x','x'],ext_field=h,field_direction='z')


hamiltonian.printout()

print(hamiltonian.abstract2qutip()[0])


Coupling Term: 

[((0, 1), {'coupling': -1.0, 'direction': ['x', 'x']}), ((1, 2), {'coupling': -1.0, 'direction': ['x', 'x']}), ((2, 0), {'coupling': -1.0, 'direction': ['x', 'x']})] 

External field: 

[((0,), {'coupling': 1.0, 'direction': ['z']}), ((1,), {'coupling': 1.0, 'direction': ['z']}), ((2,), {'coupling': 1.0, 'direction': ['z']})] 

Quantum object: dims = [[2, 2, 2], [2, 2, 2]], shape = (8, 8), type = oper, isherm = True
Qobj data =
[[ 3.  0.  0. -1.  0. -1. -1.  0.]
 [ 0.  1. -1.  0. -1.  0.  0. -1.]
 [ 0. -1.  1.  0. -1.  0.  0. -1.]
 [-1.  0.  0. -1.  0. -1. -1.  0.]
 [ 0. -1. -1.  0.  1.  0.  0. -1.]
 [-1.  0.  0. -1.  0. -1. -1.  0.]
 [-1.  0.  0. -1.  0. -1. -1.  0.]
 [ 0. -1. -1.  0. -1.  0.  0. -3.]]


#### Look at the Limbladian

In [47]:
l=3
n_dissipative=2
couplings=[1,1]
directions=['+','+']
index=[(0,),(1,)]

#create the list of dissipators
cs=[]
for i in range(n_dissipative):
    # for each site
    print(index[i],directions[i],couplings[i])
    cs.append(AbstractOperator([index[i]],direction=[directions[i]],coupling=[couplings[i]],size=l))

stdsolver=SteadyStateSolver(hamiltonian=hamiltonian,dissipative_ops=cs)
    
stdsolver.print_liouvillian()
stdsolver.get_steady_state()

print(stdsolver.steady_state)

(0,) + 1
(1,) + 1
Unitary part=

Coupling Term: 

[((0, 1), {'coupling': -1.0, 'direction': ['x', 'x']}), ((1, 2), {'coupling': -1.0, 'direction': ['x', 'x']}), ((2, 0), {'coupling': -1.0, 'direction': ['x', 'x']})] 

External field: 

[((0,), {'coupling': 1.0, 'direction': ['z']}), ((1,), {'coupling': 1.0, 'direction': ['z']}), ((2,), {'coupling': 1.0, 'direction': ['z']})] 



Dissipative part=

{(0,): {'coupling': 1, 'direction': '+'}}
{(1,): {'coupling': 1, 'direction': '+'}}


Quantum object: dims = [[2, 2, 2], [2, 2, 2]], shape = (8, 8), type = oper, isherm = True
Qobj data =
[[ 0.75298545+0.00000000e+00j  0.        +0.00000000e+00j
   0.        +0.00000000e+00j -0.11703888-5.80614819e-03j
   0.        +0.00000000e+00j -0.11703888-5.80614819e-03j
  -0.10915061-2.37127574e-02j  0.        +0.00000000e+00j]
 [ 0.        +0.00000000e+00j  0.0754919 +0.00000000e+00j
  -0.01089783-1.58777406e-02j  0.        +0.00000000e+00j
  -0.01089783-1.58777406e-02j  0.        +0.00000000e+00j
   0